# Day 9: FastAPI + MongoDB and JavaScript Basics
**CRUD APIs, aggregation, pagination and indexes, then JavaScript fundamentals**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi httpx mongomock pymongo
import mongomock
client = mongomock.MongoClient()

# ===== Option B: VS Code with real MongoDB (uncomment to use) =====
# pip install fastapi uvicorn httpx pymongo      (in your terminal)
# import os
# from pymongo import MongoClient
# client = MongoClient(os.environ.get("MONGO_URI", "mongodb://localhost:27017"))

db = client["day9"]
products, orders = db["products"], db["orders"]
products.delete_many({}); orders.delete_many({})
print("Ready")

## 2. Seed sample data

In [ ]:
import random
random.seed(9)
cats = ["electronics", "home", "stationery", "books"]
products.insert_many([
    {"sku": f"P-{i:03d}", "name": f"Product {i}", "category": random.choice(cats),
     "price": random.randint(50, 5000), "stock": random.randint(0, 40)}
    for i in range(1, 46)
])
orders.insert_many([
    {"category": random.choice(cats), "amount": random.randint(100, 9000),
     "status": random.choice(["paid", "paid", "paid", "pending", "cancelled"])}
    for _ in range(120)
])
print(products.count_documents({}), "products |", orders.count_documents({}), "orders")

## 3. Pagination

In [ ]:
def page(n, size=10):
    total = products.count_documents({})
    cur = products.find({}, {"_id": 0, "sku": 1, "name": 1}).sort("sku", 1).skip((n - 1) * size).limit(size)
    return {"page": n, "size": size, "total": total, "total_pages": (total + size - 1) // size, "items": list(cur)}

p1 = page(1)
print(p1["total_pages"], [x["sku"] for x in p1["items"]])
print([x["sku"] for x in page(5)["items"]])

# Cursor style: everything after the last sku we saw
last = p1["items"][-1]["sku"]
nxt = products.find({"sku": {"$gt": last}}, {"_id": 0, "sku": 1}).sort("sku", 1).limit(3)
print("After", last, "->", [x["sku"] for x in nxt])

## 4. Aggregation

In [ ]:
pipeline = [
    {"$match": {"status": "paid"}},
    {"$group": {"_id": "$category", "revenue": {"$sum": "$amount"},
                "orders": {"$sum": 1}, "avg_order": {"$avg": "$amount"}}},
    {"$sort": {"revenue": -1}},
    {"$project": {"_id": 0, "category": "$_id", "revenue": 1, "orders": 1, "avg_order": 1}},
]
for row in orders.aggregate(pipeline):
    print(row)

In [ ]:
low_stock = products.aggregate([
    {"$match": {"stock": {"$lt": 5}}},
    {"$group": {"_id": "$category", "items": {"$push": "$name"}, "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
])
for row in low_stock:
    print(row["_id"], row["count"], row["items"][:3])

### TODO 1: Write a pipeline that returns the highest product price in each category using `$max`.

In [ ]:
# TODO: write your code here

## 5. Indexes

In [ ]:
from pymongo.errors import DuplicateKeyError
products.create_index([("category", 1), ("price", -1)])
products.create_index("sku", unique=True)
print(list(products.index_information()))

try:
    products.insert_one({"sku": "P-001", "name": "Duplicate"})
except DuplicateKeyError:
    print("Duplicate sku rejected")

# ===== Option B only (real MongoDB): see whether a query used an index =====
# plan = products.find({"category": "home"}).sort("price", -1).explain()
# print(plan["queryPlanner"]["winningPlan"])

## 6. The products API

In [ ]:
%%writefile products_api.py
import os
from bson import ObjectId
from fastapi import Depends, FastAPI, HTTPException, Query
from pydantic import BaseModel, Field

if os.environ.get("USE_MOCK", "1") == "1":
    import mongomock
    _client = mongomock.MongoClient()
else:
    from pymongo import MongoClient
    _client = MongoClient(os.environ.get("MONGO_URI", "mongodb://localhost:27017"))
_db = _client["shop_api"]

app = FastAPI(title="Products API")

def products():
    return _db["products"]

class ProductIn(BaseModel):
    name: str = Field(min_length=1)
    category: str
    price: float = Field(gt=0)
    stock: int = Field(ge=0)

class ProductPatch(BaseModel):
    name: str | None = None
    price: float | None = Field(default=None, gt=0)
    stock: int | None = Field(default=None, ge=0)

def to_out(doc):
    doc["id"] = str(doc.pop("_id"))
    return doc

def oid(pid):
    if not ObjectId.is_valid(pid):
        raise HTTPException(404, "Product not found")
    return ObjectId(pid)

@app.post("/products", status_code=201)
def create(p: ProductIn, col=Depends(products)):
    doc = p.model_dump()
    col.insert_one(doc)
    return to_out(doc)

@app.get("/products")
def list_products(page: int = Query(1, ge=1), size: int = Query(10, ge=1, le=50),
                  category: str | None = None, col=Depends(products)):
    flt = {"category": category} if category else {}
    total = col.count_documents(flt)
    cur = col.find(flt).sort("_id", 1).skip((page - 1) * size).limit(size)
    return {"page": page, "size": size, "total": total, "items": [to_out(d) for d in cur]}

@app.get("/products/{pid}")
def get_one(pid: str, col=Depends(products)):
    doc = col.find_one({"_id": oid(pid)})
    if not doc:
        raise HTTPException(404, "Product not found")
    return to_out(doc)

@app.patch("/products/{pid}")
def patch(pid: str, data: ProductPatch, col=Depends(products)):
    changes = data.model_dump(exclude_unset=True)
    res = col.update_one({"_id": oid(pid)}, {"$set": changes})
    if res.matched_count == 0:
        raise HTTPException(404, "Product not found")
    return to_out(col.find_one({"_id": oid(pid)}))

@app.delete("/products/{pid}", status_code=204)
def delete(pid: str, col=Depends(products)):
    if col.delete_one({"_id": oid(pid)}).deleted_count == 0:
        raise HTTPException(404, "Product not found")

@app.get("/stats/by-category")
def stats(col=Depends(products)):
    rows = col.aggregate([
        {"$group": {"_id": "$category", "count": {"$sum": 1}, "avg_price": {"$avg": "$price"}}},
        {"$sort": {"count": -1}},
    ])
    return [{"category": r["_id"], "count": r["count"], "avg_price": round(r["avg_price"], 2)} for r in rows]

In [ ]:
# ===== Option A: Google Colab (active) =====
import os, importlib
os.environ["USE_MOCK"] = "1"
import products_api; importlib.reload(products_api)
from fastapi.testclient import TestClient
api = TestClient(products_api.app)

# ===== Option B: VS Code =====
# Terminal: set USE_MOCK=0 and MONGO_URI, then: uvicorn products_api:app --reload

for i in range(12):
    api.post("/products", json={"name": f"Item {i}", "category": ["home", "books"][i % 2], "price": 100 + i * 10, "stock": i})
pid = api.get("/products", params={"size": 1}).json()["items"][0]["id"]
print("Page 2:", [p["name"] for p in api.get("/products", params={"page": 2, "size": 5}).json()["items"]])
print("Patch:", api.patch(f"/products/{pid}", json={"price": 999}).json()["price"])
print("Bad id:", api.get("/products/abc").status_code)
print("Stats:", api.get("/stats/by-category").json())
print("Delete:", api.delete(f"/products/{pid}").status_code)

## 7. JavaScript basics with Node.js
Colab has Node.js installed. `%%writefile` saves the file and `!node` runs it.

In VS Code: install Node.js (LTS) from nodejs.org, save the file, then run `node basics.js` in the terminal.

In [ ]:
%%writefile basics.js
// Variables
const city = "Pune";
let count = 3;
count += 1;

// Functions
function greet(name) {
  return `Hello, ${name}! Welcome to Day 9.`;
}
const toF = (c) => c * 9 / 5 + 32;
const isEven = (n) => n % 2 === 0;

console.log(greet("Aiko"));
console.log(`37 C = ${toF(37).toFixed(1)} F`);
console.log(`7 is ${isEven(7) ? "even" : "odd"}`);

// Equality gotcha
console.log("5" == 5, "5" === 5);

In [ ]:
# ===== Option A: Google Colab (active) =====
!node basics.js

# ===== Option B: VS Code =====
# Run in the terminal:  node basics.js

In [ ]:
%%writefile arrays.js
const products = [
  { name: "Pen", price: 20, stock: 100 },
  { name: "Mug", price: 250, stock: 12 },
  { name: "Lamp", price: 1200, stock: 7 },
  { name: "Monitor", price: 9000, stock: 3 },
];

const cheap = products.filter(p => p.price < 500).map(p => p.name);
const stockValue = products.reduce((sum, p) => sum + p.price * p.stock, 0);
const priciest = products.reduce((max, p) => (p.price > max.price ? p : max));
const { name, price } = priciest;

console.log("Under 500:", cheap.join(", "));
console.log("Stock value:", stockValue);
console.log(`Most expensive: ${name} (${price})`);

for (const p of products) {
  if (p.stock < 10) console.log(`Low stock: ${p.name}`);
}

In [ ]:
# ===== Option A: Google Colab (active) =====
!node arrays.js

# ===== Option B: VS Code =====
# Run in the terminal:  node arrays.js

### TODO 2: Write `groupBy(items, key)` in a new file `group.js` and print the products grouped by whether stock is below 10.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)